# Model Equations and Notation

The equations of the models in `cas_models`, in the notation of the code
and its docstrings:

1. State-space models (continuous and discrete time)
2. Discrete-time transfer functions (`StateSpaceModelDTTFSISO`)
3. ARX models (`SSModelDTARXSISO`)
4. Kalman filters (`cas_models.discrete_time.kalman`)

## 1. State-space models

**Continuous time** (`StateSpaceModelCT`):

$$\frac{d\mathbf{x}(t)}{dt} = f(t, \mathbf{x}(t), \mathbf{u}(t), \mathbf{p})$$

$$\mathbf{y}(t) = h(t, \mathbf{x}(t), \mathbf{u}(t), \mathbf{p})$$

with $n$ state variables $\mathbf{x}$, $n_u$ input signals $\mathbf{u}$, $n_y$ output
signals $\mathbf{y}$ and parameters $\mathbf{p}$ (named `params` in the code) and $t$ 
is the time variable. In the code, `f` and `h` are CasADi functions with inputs 
`(t, x, u, *params)` and outputs named `rhs` and `y`.

**Discrete time** (`StateSpaceModelDT`):

$$\mathbf{x}(k+1) = F(t, \mathbf{x}(k), \mathbf{u}(k), \mathbf{p})$$

$$\mathbf{y}(k) = H(t, \mathbf{x}(k), \mathbf{u}(k), \mathbf{p})$$

where $k$ is the time step and $t$ the time at step $k$ (e.g. $t = k\,\Delta t$,
with $\Delta t$ the model's sample time `dt`). `F` and `H` have inputs
`(t, xk, uk, *params)` and outputs `xkp1` and `yk`. A continuous-time model
is discretized with e.g. `StateSpaceModelDTFromCTRK4` (one Runge-Kutta 4 step
per sample).

Note: here $F$ and $H$ are (possibly nonlinear) functions, not matrices. Matrices 
$\mathbf{A}, \mathbf{B}, \mathbf{C}, \mathbf{D}$ are used to define linear models.

**Linear models** (`StateSpaceModelCTFromABCD`, `StateSpaceModelDTFromABCD`):

$$\frac{d\mathbf{x}(t)}{dt} = \mathbf{A}\mathbf{x}(t) + \mathbf{B}\mathbf{u}(t), \quad
\mathbf{y}(t) = \mathbf{C}\mathbf{x}(t) + \mathbf{D}\mathbf{u}(t)$$

$$\mathbf{x}(k+1) = \mathbf{A}\mathbf{x}(k) + \mathbf{B}\mathbf{u}(k), \quad
\mathbf{y}(k) = \mathbf{C}\mathbf{x}(k) + \mathbf{D}\mathbf{u}(k)$$

`linearize(sys, x, u)` gives the matrices of any model about $(\mathbf{x}, \mathbf{u})$:
the Jacobians $\mathbf{A} = \partial f / \partial \mathbf{x}$ (or $\partial F / \partial \mathbf{x}$),
$\mathbf{B} = \partial f / \partial \mathbf{u}$, $\mathbf{C} = \partial h / \partial \mathbf{x}$,
$\mathbf{D} = \partial h / \partial \mathbf{u}$.

A model has direct transmission (feedthrough) when $\mathbf{y}(k)$ depends on
$\mathbf{u}(k)$ ($\mathbf{D} \neq 0$).

## 2. Discrete-time transfer functions

The `StateSpaceModelDTTFSISO` class implements a discrete-time transfer function model:

$$G(z) = \frac{B(z)}{A(z)} = \frac{b_0 z^n + b_1 z^{n-1} + \cdots + b_n}{z^n + a_1 z^{n-1} + \cdots + a_n}$$

where $z$ is the forward shift operator in the z-transform domain. The
coefficients are given in descending powers of $z$, `num` $= [b_0, \ldots]$
and `den` $= [a_0, a_1, \ldots, a_n]$; the class divides both by $a_0$ (so
$a_0 = 1$ above) and pads a shorter numerator with leading zeros ($b_0 = 0$,
$b_1 = 0$, ...: a delay).

Note the numbering here: $b_0, \ldots, b_n$ from the powers of $z$, is not the same as
it is in $b_1, \ldots, b_{nb}$ in ARX models (section 3), from the powers of $q^{-1}$.

This translates to the following difference equation:

$$y(k) = -a_1 y(k-1) - a_2 y(k-2) - \cdots - a_n y(k-n) + b_0 u(k) + b_1 u(k-1) + \cdots + b_n u(k-n)$$

The model is internally converted to state-space form using the observable canonical form representation.

Example:

From `den = [1 -2.3 2.6 -1.8 0.72 -0.12]`, we have:
  - $a_1 = -2.3, a_2 = 2.6, a_3 = -1.8, a_4 = 0.72, a_5 = -0.12$

From `num = [0.01 0.03 0.03 0.02 0.01]`, padded with one leading zero to
the denominator's length (a one-step delay), we have:
  - $b_0 = 0, b_1 = 0.01, b_2 = 0.03, b_3 = 0.03, b_4 = 0.02, b_5 = 0.01$

The state-space model in observable canonical form is:

$$\mathbf{x}(k+1) = \begin{bmatrix}
0 & 0 & 0 & 0 & -a_5 \\
1 & 0 & 0 & 0 & -a_4 \\
0 & 1 & 0 & 0 & -a_3 \\
0 & 0 & 1 & 0 & -a_2 \\
0 & 0 & 0 & 1 & -a_1 \\
\end{bmatrix} \mathbf{x}(k) + \begin{bmatrix}
b_5 \\ b_4 \\ b_3 \\ b_2 \\ b_1
\end{bmatrix} u(k)$$

$$y(k) = \begin{bmatrix} 0 & 0 & 0 & 0 & 1 \end{bmatrix} \mathbf{x}(k)$$

($D = b_0 = 0$).

Substituting the values:

$$\mathbf{x}(k+1) = \begin{bmatrix}
0 & 0 & 0 & 0 & 0.12 \\
1 & 0 & 0 & 0 & -0.72 \\
0 & 1 & 0 & 0 & 1.8 \\
0 & 0 & 1 & 0 & -2.6 \\
0 & 0 & 0 & 1 & 2.3 \\
\end{bmatrix} \mathbf{x}(k) + \begin{bmatrix}
0.01 \\ 0.02 \\ 0.03 \\ 0.03 \\ 0.01
\end{bmatrix} u(k)$$

**General form for transfer function to observable canonical state-space:**

Given a discrete-time transfer function:

$$G(z) = \frac{b_0 z^n + b_1 z^{n-1} + \cdots + b_n}{z^n + a_1 z^{n-1} + \cdots + a_n}$$

The observable canonical form state-space representation is:

$$\mathbf{x}(k+1) = \begin{bmatrix}
0 & 0 & \cdots & 0 & -a_n \\
1 & 0 & \cdots & 0 & -a_{n-1} \\
0 & 1 & \cdots & 0 & -a_{n-2} \\
\vdots & \vdots & \ddots & \vdots & \vdots \\
0 & 0 & \cdots & 1 & -a_1 \\
\end{bmatrix} \mathbf{x}(k) + \begin{bmatrix}
b_n - a_n b_0 \\ b_{n-1} - a_{n-1} b_0 \\ \vdots \\ b_1 - a_1 b_0
\end{bmatrix} u(k)$$

$$y(k) = \begin{bmatrix} 0 & 0 & \cdots & 0 & 1 \end{bmatrix} \mathbf{x}(k) + b_0 \, u(k)$$

So $\mathbf{D} = b_0$, which is zero for a strictly proper transfer function
(numerator of lower degree than the denominator), and then
$\mathbf{B} = [b_n, \ldots, b_1]^T$.
(Before 2026-11-16, the code used $\mathbf{B} = [b_n, \ldots, b_1]^T$ also when
$b_0 \neq 0$, which gave a different transfer function.)

## 3. ARX models

The `SSModelDTARXSISO` class implements an ARX model:

$$A(q^{-1}) y(k) = B(q^{-1}) q^{-nk} u(k) + e(k)$$

where:
- $A(q^{-1}) = 1 + a_1 q^{-1} + a_2 q^{-2} + \cdots + a_{na} q^{-na}$
- $B(q^{-1}) = b_1 + b_2 q^{-1} + \cdots + b_{nb} q^{-nb+1}$
- $q^{-1}$ is the backward shift operator, $q^{-1} y(k) = y(k-1)$
- $nk$ is the input delay (time steps), and $e(k)$ white noise

Note that $A$ and $B$ are polynomials here, not the state-space matrices.

The coefficients are numbered from 1, as are the model's parameters
(`a_1`, ..., `b_1`, ...). This is the convention of MATLAB's `arx()` and
`filter()` and SciPy's `lfilter()` (Octave's `arx()` counts `nk` one less;
see the class docstring).

This translates to the difference equation:

$$y(k) = -a_1 y(k-1) - \cdots - a_{na} y(k-na) + b_1 u(k-nk) + b_2 u(k-nk-1) + \cdots + b_{nb} u(k-nk-nb+1) + e(k)$$

ARX is the simplest of the prediction error model structures,

$$y(k) = \frac{B(q^{-1})\, q^{-nk}}{A(q^{-1}) F(q^{-1})} u(k) + \frac{C(q^{-1})}{A(q^{-1}) D(q^{-1})} e(k)$$

with $C = D = F = 1$.

With `poles` (or `real_poles=True`), $A$ is parametrized by its roots,
$A(q^{-1}) = (1 - p_1 q^{-1}) \cdots (1 - p_{na} q^{-1})$, and with `zeros`,
$B(q^{-1}) = b_1 (1 - z_1 q^{-1}) \cdots (1 - z_{nb-1} q^{-1})$.

**State-space representation:**

The model is simulated without the noise ($e(k) = 0$). Its transfer function
is that of section 2: multiplying by $z^N$, with $N = \max(na, nk + nb - 1)$,

$$G(z) = \frac{z^{N-nk-nb+1} \left(b_1 z^{nb-1} + b_2 z^{nb-2} + \cdots + b_{nb}\right)}
{z^N + a_1 z^{N-1} + \cdots + a_{na} z^{N-na}}$$

that is, `den` $= [1, a_1, \ldots, a_{na}, 0, \ldots]$ and
`num` $= [0, \ldots, 0, b_1, \ldots, b_{nb}, 0, \ldots]$ (with $nk$ leading
zeros), both of length $N + 1$ (`arx_tf_coefficients()`). There are two
realizations of it (`realization=`), the same model apart from the states:

- `"minimal"` (the default): the observable canonical form of section 2,
  with $n = N = \max(na, nk + nb - 1)$ states. It is controllable and
  observable unless $A$ and $B$ have a common root (a pole-zero
  cancellation, which depends on the parameters' values). The states aren't
  past signals, so an initial state has to be estimated, but every part of
  it (and of a Kalman filter gain) affects the output.
- `"shift"`: a shift register of the past outputs and inputs (below), with
  $n = na + nk + nb - 1$ states. They are easy to interpret and to
  initialize from data, but when $nk + nb > 1$ the realization isn't
  minimal: it has $\min(na, nk + nb - 1)$ extra poles at $z = 0$ that the
  output doesn't show, so an estimated initial state, or a Kalman filter
  gain fitted to data, has combinations the data can't determine.

The orders are $na \ge 1$, $nb \ge 1$ and $nk \ge 0$.

**Minimal realization**

**Example: ARX(2, 2, 1)**

$G(z) = \dfrac{b_1 z + b_2}{z^2 + a_1 z + a_2}$, $n = 2$:

$$\mathbf{A} = \begin{bmatrix} 0 & -a_2 \\ 1 & -a_1 \end{bmatrix}, \quad
\mathbf{B} = \begin{bmatrix} b_2 \\ b_1 \end{bmatrix}, \quad
\mathbf{C} = \begin{bmatrix} 0 & 1 \end{bmatrix}, \quad
\mathbf{D} = 0$$

**Example: ARX(2, 2, 0)** (direct transmission)

$G(z) = \dfrac{b_1 z^2 + b_2 z}{z^2 + a_1 z + a_2}$, $n = 2$, and with
$b_0 = b_1$ in section 2's general $\mathbf{B}$:

$$\mathbf{A} = \begin{bmatrix} 0 & -a_2 \\ 1 & -a_1 \end{bmatrix}, \quad
\mathbf{B} = \begin{bmatrix} -a_2 b_1 \\ b_2 - a_1 b_1 \end{bmatrix}, \quad
\mathbf{C} = \begin{bmatrix} 0 & 1 \end{bmatrix}, \quad
\mathbf{D} = b_1$$

**Shift-register realization**

The states are the past outputs and inputs:

$$\mathbf{x}(k) = [y(k-1), \ldots, y(k-na), u(k-1), \ldots, u(k-m)]^T,
\quad m = nk + nb - 1$$

so there are $n = na + m$ states.
Writing $\beta_j$ for the coefficient of $u(k-j)$ in the difference equation
($\beta_j = b_{j-nk+1}$ for $nk \le j \le m$, else 0):

$$y(k) = -a_1 y(k-1) - \cdots - a_{na} y(k-na) + \beta_1 u(k-1) + \cdots + \beta_m u(k-m) + \beta_0 u(k)$$

which is $y(k) = \mathbf{C}\mathbf{x}(k) + \mathbf{D}u(k)$ with

$$\mathbf{C} = \begin{bmatrix} -a_1 & \cdots & -a_{na} & \beta_1 & \cdots & \beta_m \end{bmatrix},
\quad \mathbf{D} = \beta_0$$

($\mathbf{D} = b_1$ when $nk = 0$, else 0). The state update shifts the
registers: the first state becomes $y(k)$, so the first row of $\mathbf{A}$ is
$\mathbf{C}$ and the first element of $\mathbf{B}$ is $\mathbf{D}$, and $u(k)$
enters the first input state.

**Example: ARX(2, 2, 1)**

For $na = 2$, $nb = 2$, $nk = 1$: $m = 2$, $n = 4$,
$\mathbf{x}(k) = [y(k-1), y(k-2), u(k-1), u(k-2)]^T$ and

$$\mathbf{A} = \begin{bmatrix}
-a_1 & -a_2 & b_1 & b_2 \\
1 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 \\
0 & 0 & 1 & 0
\end{bmatrix}, \quad
\mathbf{B} = \begin{bmatrix} 0 \\ 0 \\ 1 \\ 0 \end{bmatrix}, \quad
\mathbf{C} = \begin{bmatrix} -a_1 & -a_2 & b_1 & b_2 \end{bmatrix}, \quad
\mathbf{D} = 0$$

**Example: ARX(2, 2, 0)** (direct transmission)

For $nk = 0$: $m = 1$, $n = 3$, $\mathbf{x}(k) = [y(k-1), y(k-2), u(k-1)]^T$ and

$$\mathbf{A} = \begin{bmatrix}
-a_1 & -a_2 & b_2 \\
1 & 0 & 0 \\
0 & 0 & 0
\end{bmatrix}, \quad
\mathbf{B} = \begin{bmatrix} b_1 \\ 0 \\ 1 \end{bmatrix}, \quad
\mathbf{C} = \begin{bmatrix} -a_1 & -a_2 & b_2 \end{bmatrix}, \quad
\mathbf{D} = b_1$$

## 4. Kalman filters

The discrete-time model of the plant, with process and measurement noise:

$$\mathbf{x}(k+1) = \mathbf{A}\mathbf{x}(k) + \mathbf{B}\mathbf{u}(k) + \mathbf{w}(k), \quad \mathbf{w}(k) \sim N(0, \mathbf{Q})$$

$$\mathbf{y}(k) = \mathbf{C}\mathbf{x}(k) + \mathbf{D}\mathbf{u}(k) + \mathbf{v}(k), \quad \mathbf{v}(k) \sim N(0, \mathbf{R})$$

$\hat{\mathbf{x}}(k|j)$ is the estimate of $\mathbf{x}(k)$ given the data up
to time $j$, and $\mathbf{P}(k|j)$ the covariance of its error. Names in
the code and in docstrings:

| Quantity | Meaning | Code |
|---|---|---|
| $\hat{\mathbf{x}}(k \mid k-1)$, $\mathbf{P}(k \mid k-1)$ | prior estimate at $k$ (the previous step's prediction) | `xk_pred`, `Pk_pred` |
| $\hat{\mathbf{x}}(k \mid k)$, $\mathbf{P}(k \mid k)$ | posterior (corrected) estimate | `xk_est`, `Pk` |
| $\hat{\mathbf{x}}(k+1 \mid k)$, $\mathbf{P}(k+1 \mid k)$ | prediction for $k+1$ | `xkp1_est`, `Pkp1` |
| $\hat{\mathbf{y}}(k \mid k-1)$ | prediction of $\mathbf{y}(k)$ before it is measured | `yk_pred` |
| $\hat{\mathbf{y}}(k \mid k)$, $\hat{\mathbf{y}}(k+1 \mid k)$ | output estimates | `yk_est`, `ykp1_est` |
| $\mathbf{S}(k \mid k-1)$ | covariance of the innovations | `Sk_pred` |
| $\mathbf{K}_f(k)$ | correction gain (filtering form) | `Kf` |
| $\mathbf{K}_p(k) = \mathbf{A}\mathbf{K}_f(k)$ | correction gain (prediction form) | `Kp`, `K` in `KFPSS` |

**Correction** (`kalman_update_f`), with the measurement $\mathbf{y}(k)$:

$$\hat{\mathbf{y}}(k|k-1) = \mathbf{C}\hat{\mathbf{x}}(k|k-1) + \mathbf{D}\mathbf{u}(k)$$

$$\mathbf{S}(k|k-1) = \mathbf{C}\mathbf{P}(k|k-1)\mathbf{C}^T + \mathbf{R}$$

$$\mathbf{K}_f(k) = \mathbf{P}(k|k-1)\mathbf{C}^T \mathbf{S}(k|k-1)^{-1}$$

$$\hat{\mathbf{x}}(k|k) = \hat{\mathbf{x}}(k|k-1) + \mathbf{K}_f(k)\left(\mathbf{y}(k) - \hat{\mathbf{y}}(k|k-1)\right)$$

$$\mathbf{P}(k|k) = \left(\mathbf{I} - \mathbf{K}_f(k)\mathbf{C}\right)\mathbf{P}(k|k-1)\left(\mathbf{I} - \mathbf{K}_f(k)\mathbf{C}\right)^T + \mathbf{K}_f(k)\mathbf{R}\mathbf{K}_f(k)^T$$

(the Joseph form of $\mathbf{P}(k|k) = \mathbf{P}(k|k-1) - \mathbf{K}_f(k)\mathbf{S}(k|k-1)\mathbf{K}_f(k)^T$,
which keeps $\mathbf{P}$ positive semi-definite despite rounding errors).
$\mathbf{e}(k) = \mathbf{y}(k) - \hat{\mathbf{y}}(k|k-1)$ are the innovations,
with covariance $\mathbf{S}(k|k-1)$.

**Prediction** (`kalman_predict_f`), with the input $\mathbf{u}(k)$:

$$\hat{\mathbf{x}}(k+1|k) = \mathbf{A}\hat{\mathbf{x}}(k|k) + \mathbf{B}\mathbf{u}(k)$$

$$\mathbf{P}(k+1|k) = \mathbf{A}\mathbf{P}(k|k)\mathbf{A}^T + \mathbf{Q}$$

`kf_update` does both, a complete time step. The filters' output estimates
and their error variances:

$$\hat{\mathbf{y}}(k|k) = \mathbf{C}\hat{\mathbf{x}}(k|k) + \mathbf{D}\mathbf{u}(k), \quad
\hat{\mathbf{y}}(k+1|k) = \mathbf{C}\hat{\mathbf{x}}(k+1|k) \;(\mathbf{D} = 0 \text{ only})$$

$$\operatorname{diag}\left(\mathbf{C}\mathbf{P}(k|k)\mathbf{C}^T\right), \quad
\operatorname{diag}\left(\mathbf{C}\mathbf{P}(k+1|k)\mathbf{C}^T\right)$$

the variances of the errors in estimating the noise-free outputs (without
$\mathbf{R}$; the error in predicting the measurement $\mathbf{y}(k+1)$ has
covariance $\mathbf{C}\mathbf{P}(k+1|k)\mathbf{C}^T + \mathbf{R}$).

**Prediction and filtering forms.** `KFP` outputs $\hat{\mathbf{x}}(k+1|k)$,
`KFF` outputs $\hat{\mathbf{x}}(k|k)$; both have the states
$\hat{\mathbf{x}}(k|k-1)$ and $\mathbf{P}(k|k-1)$. Combining the two steps,
the prediction form is

$$\hat{\mathbf{x}}(k+1|k) = \mathbf{A}\hat{\mathbf{x}}(k|k-1) + \mathbf{B}\mathbf{u}(k) + \mathbf{K}_p(k)\left(\mathbf{y}(k) - \hat{\mathbf{y}}(k|k-1)\right), \quad
\mathbf{K}_p(k) = \mathbf{A}\mathbf{K}_f(k)$$

**Steady-state filters.** $\mathbf{P} = \mathbf{P}(k+1|k)$ solves the discrete
algebraic Riccati equation (`steady_state_kf_gain`)

$$\mathbf{P} = \mathbf{A}\mathbf{P}\mathbf{A}^T - \mathbf{A}\mathbf{P}\mathbf{C}^T\left(\mathbf{C}\mathbf{P}\mathbf{C}^T + \mathbf{R}\right)^{-1}\mathbf{C}\mathbf{P}\mathbf{A}^T + \mathbf{Q}$$

and the gains are constant, $\mathbf{K}_f = \mathbf{P}\mathbf{C}^T(\mathbf{C}\mathbf{P}\mathbf{C}^T + \mathbf{R})^{-1}$,
$\mathbf{K}_p = \mathbf{A}\mathbf{K}_f$. `KFPSS` is the innovations form, a
linear system with gain $\mathbf{K} = \mathbf{K}_p$:

$$\hat{\mathbf{x}}(k+1|k) = (\mathbf{A} - \mathbf{K}\mathbf{C})\hat{\mathbf{x}}(k|k-1) + (\mathbf{B} - \mathbf{K}\mathbf{D})\mathbf{u}(k) + \mathbf{K}\mathbf{y}(k)$$

and `KFFSS` (gain $\mathbf{K} = \mathbf{K}_f$) also outputs

$$\hat{\mathbf{x}}(k|k) = (\mathbf{I} - \mathbf{K}_f\mathbf{C})\hat{\mathbf{x}}(k|k-1) - \mathbf{K}_f\mathbf{D}\mathbf{u}(k) + \mathbf{K}_f\mathbf{y}(k)$$

**Extended Kalman filters** (`EKFP`, `EKFF`, one step: `ekf_update`), for a
model $\mathbf{x}(k+1) = F(t, \mathbf{x}(k), \mathbf{u}(k)) + \mathbf{w}(k)$,
$\mathbf{y}(k) = H(t, \mathbf{x}(k), \mathbf{u}(k)) + \mathbf{v}(k)$:
the same equations, with the functions in place of the linear terms and their
Jacobians (by automatic differentiation) about the latest estimates:

$$\mathbf{C}(k) = \left.\frac{\partial H}{\partial \mathbf{x}}\right|_{\hat{\mathbf{x}}(k|k-1),\, \mathbf{u}(k)}, \quad
\hat{\mathbf{y}}(k|k-1) = H(t, \hat{\mathbf{x}}(k|k-1), \mathbf{u}(k))$$

in the correction, and

$$\mathbf{A}(k) = \left.\frac{\partial F}{\partial \mathbf{x}}\right|_{\hat{\mathbf{x}}(k|k),\, \mathbf{u}(k)}, \quad
\hat{\mathbf{x}}(k+1|k) = F(t, \hat{\mathbf{x}}(k|k), \mathbf{u}(k)), \quad
\mathbf{P}(k+1|k) = \mathbf{A}(k)\mathbf{P}(k|k)\mathbf{A}(k)^T + \mathbf{Q}$$

in the prediction. With a linear model, this is the Kalman filter above.